In [24]:
import zstandard as zstd
import pandas as pd
import json
import io
import os
from datetime import datetime
from polygon import RESTClient
import os
from dotenv import load_dotenv
load_dotenv()
polygon_key = os.getenv("POLYGON_API_KEY")
client = RESTClient(polygon_key)

In [7]:
def parse_reddit_zst(input_filepath, output_filepath, is_submission=True):
    """
    Reads a pre-split Reddit .zst file, extracts the core text and timestamps,
    and saves it as a compressed Parquet file for Machine Learning.
    """
    print(f"Opening {input_filepath}...")
    
    records = []
    dctx = zstd.ZstdDecompressor(max_window_size=2147483648)
    
    with open(input_filepath, 'rb') as fh:
        with dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8')
            
            for i, line in enumerate(text_stream):
                try:
                    obj = json.loads(line)
                    
                    record = {
                        'datetime': datetime.utcfromtimestamp(int(obj['created_utc'])),
                        'score': obj.get('score', 0),
                        'upvote_ratio': obj.get('upvote_ratio', 0.0),
                        'flair': obj.get('link_flair_text', ''),
                    }
                    
                    if is_submission:
                        title = obj.get('title', '')
                        selftext = obj.get('selftext', '')
                        record['text'] = f"{title}. {selftext}".strip()
                    else:
                        record['text'] = obj.get('body', '').strip()
                    
                    if record['text'] not in ['[deleted]', '[removed]', '.', '']:
                        records.append(record)
                        
                except Exception as e:
                    continue
                
                # Print update to the terminal every 100,000 lines 
                if (i + 1) % 100000 == 0:
                    print(f"Processed {i + 1:,} lines... (Found {len(records):,} valid records)")

    print(f"\nConverting {len(records):,} records to Pandas DataFrame...")
    df = pd.DataFrame(records)
    df.to_parquet(output_filepath, engine='pyarrow', index=False)
    print("complete")

if __name__ == "__main__":
    
    INPUT_FILE = "wallstreetbets_submissions.zst" 
    OUTPUT_FILE = "wsb_submissions_clean.parquet"
    
    if os.path.exists(INPUT_FILE):
        parse_reddit_zst(INPUT_FILE, OUTPUT_FILE, is_submission=True)
    else:
        print(f"Error: Could not find '{INPUT_FILE}'")

Opening wallstreetbets_submissions.zst...
Processed 100,000 lines... (Found 100,000 valid records)
Processed 200,000 lines... (Found 200,000 valid records)
Processed 300,000 lines... (Found 300,000 valid records)
Processed 400,000 lines... (Found 400,000 valid records)
Processed 500,000 lines... (Found 500,000 valid records)
Processed 600,000 lines... (Found 600,000 valid records)
Processed 700,000 lines... (Found 700,000 valid records)
Processed 800,000 lines... (Found 800,000 valid records)
Processed 900,000 lines... (Found 900,000 valid records)
Processed 1,000,000 lines... (Found 1,000,000 valid records)
Processed 1,100,000 lines... (Found 1,100,000 valid records)
Processed 1,200,000 lines... (Found 1,200,000 valid records)
Processed 1,300,000 lines... (Found 1,300,000 valid records)
Processed 1,400,000 lines... (Found 1,400,000 valid records)
Processed 1,500,000 lines... (Found 1,500,000 valid records)
Processed 1,600,000 lines... (Found 1,600,000 valid records)
Processed 1,700,00

In [ ]:
'''Why Parquet instead of CSV?
Speed & Size: CSV is a "row-based" plain text format. 
Parquet is a "columnar" binary format. A 10GB CSV file will often compress down to a 1GB Parquet file. 
Pandas can read it up to 50x faster.

Data Types are Saved: 
In a CSV, datetime columns are saved as plain text. 
Every time you load the CSV, Pandas has to waste time recalculating and parsing those strings back into dates. 
Parquet physically saves the underlying data types (dates stay dates, integers stay integers), meaning it loads instantly into memory
'''

In [5]:
def inspect_parquet_data(filepath):
    print(f"Loading {filepath}...\n")
    df = pd.read_parquet(filepath)
    
    print(f"dataset contains {len(df):,} posts.\n")
    
    print("--- DATA OVERVIEW ---")
    print(df.info())
    
    print("First 3 Posts")
    pd.set_option('display.max_colwidth', None) 
    print(df[['datetime', 'score', 'text']].head(3))

    print("\n--- RANDOM POST ---")
    random_post = df.sample(1)
    print(f"Date: {random_post['datetime'].values[0]}")
    print(f"Score: {random_post['score'].values[0]}")
    print(f"Text: {random_post['text'].values[0]}")

if __name__ == "__main__":
    PARQUET_FILE = "wsb_submissions_clean.parquet"
    inspect_parquet_data(PARQUET_FILE)

Loading wsb_submissions_clean.parquet...

dataset contains 2,644,070 posts.

--- DATA OVERVIEW ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644070 entries, 0 to 2644069
Data columns (total 3 columns):
 #   Column    Dtype         
---  ------    -----         
 0   datetime  datetime64[ns]
 1   score     int64         
 2   text      object        
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 60.5+ MB
None
First 3 Posts
             datetime  score  \
0 2012-04-11 16:40:40     13   
1 2012-04-12 20:37:31      2   
2 2012-04-16 22:29:37     12   

                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [29]:
import pandas as pd
import re
import nltk
import os
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from polygon import RESTClient
from dotenv import load_dotenv

tqdm.pandas()

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    print("Downloading WordNet dictionary for Lemmatization...")
    nltk.download('wordnet')
    nltk.download('omw-1.4')

load_dotenv()
polygon_key = os.getenv("POLYGON_API_KEY")
client = RESTClient(polygon_key)

# note: the letter C is blacklisted even though it is the ticker symbol of Citigroup is because it means Call option in wsb
def extract_ticker_from_text(text): 
    blacklist_words = [
      "YOLO", "TOS", "CEO", "CFO", "CTO", "DD", "BTFD", "WSB", "OK", "RH",
      "KYS", "FD", "TYS", "US", "USA", "IT", "ATH", "RIP", "BMW", "GDP",
      "OTM", "ATM", "ITM", "IMO", "LOL", "DOJ", "BE", "PR", "PC", "ICE",
      "TYS", "ISIS", "PRAY", "PT", "FBI", "SEC", "GOD", "NOT", "POS", "COD",
      "AYYMD", "FOMO", "TL;DR", "EDIT", "STILL", "LGMA", "WTF", "RAW",
      "LMAO", "LMFAO", "ROFL", "EZ", "RED", "BEZOS", "TICK", "IS", "DOW",
      "AM", "PM", "LPT", "GOAT", "FL", "CA", "IL", "PDFUA", "MACD", "HQ",
      "OP", "DJIA", "PS", "AH", "TL", "DR", "JAN", "FEB", "JUL", "AUG",
      "SEP", "SEPT", "OCT", "NOV", "DEC", "FDA", "IV", "ER", "IPO", "RISE",
      "IPA", "URL", "MILF", "BUT", "SSN", "FIFA", "USD", "CPU", "AT",
      "GG", "ELON", "I", "WE", "A", "OMG", "GUH", "HODL", "HOLD", "TLDR", "AI", 
      "DD", "TDM", "C", "B", "IV", "FD", "P", "M", "E", "S", "U", "AAA"
   ]
    matches = re.findall(r'\$?[A-Z]{1,5}\b', str(text))
    
    # Clean off the '$' so "$AAPL" and "AAPL" are treated the same
    # set() to remove duplicates and O(1) access time
    unique_tickers = list(set([m.replace('$', '') for m in matches]))
    return [t for t in unique_tickers if t not in blacklist_words]

def check_if_ticker_valid(word):
    try:
        # special case for ROPE
        if word != "ROPE":
            # sends request to Massive API to determine whether the current word is a valid ticker
            details = client.get_ticker_details(word)
            return True
    except Exception as e:
        return False
    return False

def clean_and_lemmatize(text, lemmatizer):
    """
    Cleans the text but strictly PRESERVES tickers and the $ symbol.
    """
    text = str(text)
    # remove web links 
    text = re.sub(r'http\S+', '', text)        
    
    # remove punctuation, keep letters, spaces, and $ sign
    text = re.sub(r'[^a-zA-Z\$\s]', '', text)    
    
    # LEMMATIZATION
    # Convert to lowercase and split into words
    words = text.lower().split()
    
    # Pass each word through the lemmatizer
    return ' '.join([lemmatizer.lemmatize(w) for w in words])

def filter_wsb_posts(df: pd.DataFrame) -> pd.DataFrame:
    print(f"Initial rows before validation: {len(df):,}")
    
    # fill missing strings with empty strings to prevent crash
    df['text'] = df['text'].fillna('')
    df['flair'] = df['flair'].fillna('')
    # fill missing ratios with 0
    df['upvote_ratio'] = df['upvote_ratio'].fillna(0.0)
    
    df = df[~df['text'].str.isspace()]
    
    junk_texts = ['[deleted]', '[removed]', 'deleted', 'removed', '']
    df = df[~df['text'].str.strip().str.lower().isin(junk_texts)]
    
    # score and upvote filter
    df = df[(df['score'] >= 5) & (df['upvote_ratio'] > 0.5)]
    
    # require the flair to exist
    df = df[df['flair'].str.strip() != '']
    
    valid_flairs = ['DD', 'Discussion', 'Fundamentals', 'Chart']
    df = df[df['flair'].isin(valid_flairs)]
    
    # character count filter
    df = df[df['text'].str.len() >= 75]
    
    df = df.reset_index(drop=True)
    
    print(f"Filter Complete, remaining posts: {len(df):,}")
    
    return df



[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/thibautchen/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/thibautchen/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [19]:
'''
Testing extract_ticker_from_text and check_if_ticker_valid
'''
text1 = "I am buying $AAPL and TSLA plus $F and C tomorrow, OMG AAPL is so ICE. YOLO! KKK" #end -> test check_if_ticker_valid function
print(text1)
print()
print(extract_ticker_from_text(text1))
ticker_valid_test_list = []
for word in extract_ticker_from_text(text1):
    ticker_valid_test_list.append(check_if_ticker_valid(word))
    
ticker_valid_test_list



I am buying $AAPL and TSLA plus $F and C tomorrow, OMG AAPL is so ICE. YOLO! KKK

['TSLA', 'C', 'KKK', 'F', 'AAPL']


[True, True, False, True, True]

In [20]:
'''
Testing cleaning and lemmatization
'''
lemmatizer = WordNetLemmatizer()
    
raw_text = "I am BUYING 100 shares of $NVDA!!! Check this link: https://google.com 🚀"
cleaned = clean_and_lemmatize(raw_text, lemmatizer)

expected = "i am buying share of $nvda check this link"
    
assert cleaned == expected, f"Lemmatization failed.\nExpected: {expected}\nGot: {cleaned}"
print("clean_and_lemmatize test passes\n")

clean_and_lemmatize passed all tests!



In [23]:
'''
Testing filter function
'''

mock_data = pd.DataFrame({
        'text': [
            "This is a fantastic Due Diligence post about Apple that is very long and has lots of good information.", # valid
            "[removed]", # invalid: removed
            "   ", # invalid: blank spaces
            "Short post", # invalid: under character count
            "This is a great long post but everyone hated it so it got downvoted to oblivion.", # invalid: score < 5
            "This post is long enough and has a good score, but the upvote ratio is terrible due to controversy.", # invalid: Ratio < 0.5
            "Valid post but missing a flair because the user forgot to tag it.", # invalid: No Flair
            "Valid post but its a stupid ahh meme so its a fuckiing shitpost lololol." # invalid: shitpost
        ],
        'score': [100, 10, 10, 10, 2, 100, 100, 100],
        'upvote_ratio': [0.95, 0.9, 0.9, 0.9, 0.9, 0.4, 0.9, 0.9],
        'link_flair_text': ['DD', 'DD', 'DD', 'DD', 'DD', 'DD', '', 'shitpost']
    })
    
filtered_df = filter_wsb_posts(mock_data)
    
assert len(filtered_df) == 1, f"filter failed,  expected 1 post to survive, but got {len(filtered_df)}."
assert filtered_df.iloc[0]['score'] == 100, "The wrong post survived the filter."
    
print("filter_wsb_posts passed all tests\n")

Initial rows before validation: 8
Filter Complete, remaining posts: 1
filter_wsb_posts passed all tests



In [30]:
import pandas as pd
import re
import nltk
import os
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm

tqdm.pandas()

ticker_cache = {}

def extract_ticker_from_text_with_validation_using_caching(text):
    potential_tickers = extract_ticker_from_text(text)
    valid_tickers = []
    for word in potential_tickers:
        if word in ticker_cache:
            if ticker_cache[word]:
                valid_tickers.append(word)
        else:
            if check_if_ticker_valid(word):
                ticker_cache[word] = True
                valid_tickers.append(word)
            else:
                ticker_cache[word] = False
    return valid_tickers

def process_dataset(input_file: str, output_file: str):
    print(f"loading dataset: {input_file}")
    df = pd.read_parquet(input_file)
    
    df = filter_wsb_posts(df)
    
    if len(df) == 0:
        print("ERROR: no rows left after filtering")
        return
        
    print("\n Extracting Tickers...")
    # progress_apply for a loading bar
    df['tickers'] = df['text'].progress_apply(extract_ticker_from_text_with_validation_using_caching)
    
    print("\n lemmatizing")
    lemmatizer = WordNetLemmatizer()
    df['clean_text'] = df['text'].progress_apply(lambda x: clean_and_lemmatize(x, lemmatizer))
    
    print(f"\n saving final ML-ready dataset...")
    columns_to_keep = ['datetime', 'score', 'tickers', 'clean_text', 'flair', 'upvote_ratio']
        
    final_df = df[columns_to_keep]
    final_df.to_parquet(output_file, engine='pyarrow', index=False)
    
    print(f"saved to {output_file}")


if __name__ == "__main__":
    INPUT_DATA = "wsb_submissions_clean.parquet" 
    
    OUTPUT_DATA = "wsb_submissions_ml_ready.parquet"
    
    if os.path.exists(INPUT_DATA):
        process_dataset(INPUT_DATA, OUTPUT_DATA)
    else:
        print(f"Error: cannot find {INPUT_DATA}")

loading dataset: wsb_submissions_clean.parquet
Initial rows before validation: 2,644,070
Filter Complete, remaining posts: 77,423

 Extracting Tickers...


100%|██████████| 77423/77423 [32:18<00:00, 39.94it/s]  



 lemmatizing


100%|██████████| 77423/77423 [00:24<00:00, 3102.68it/s] 



 saving final ML-ready dataset...
saved to wsb_submissions_ml_ready.parquet
